In [2]:
import pandas as pd
import time 
import requests
import json
#other requirements: sqlalchemy, pscycopg2

In [3]:
from DatabaseCreator import DatabaseCreator

In [4]:
start=time.perf_counter()

In [5]:
current_season = 20262027

In [6]:
# #get fixtures before changing code
# from Load_Merge_FPL_Data import FPLDataManager 
# import CJDH_local_settings

# #Run Database Creator
# if __name__ == "__main__":
#     manager = FPLDataManager()
#     #FPLAPIDataFetcher: id max = 5 - remove after testing
#     fpl_data, fixtures_with_elo, merged, current_year_players_allfixtures = manager.load_2025_26_season_from_api()

# current_year_players_allfixtures.head()

In [7]:
from Load_Merge_FPL_Data import FPLDataManager
import CJDH_local_settings

#Run Database Creator
if __name__ == "__main__":
    manager = FPLDataManager(current_season=current_season)
    df_all_seasons = manager.load_all_seasons()
    print(f"\nTotal records: {len(df_all_seasons)}")

    db_creator = DatabaseCreator(db_settings=CJDH_local_settings.local_settings['FPL_Points_Predictor'])
    fpl_engine = db_creator.get_engine_for("fpl_data_analysis")

    #Add a new staging table & data into the database
    playergw = df_all_seasons[['player_name_id','element','value','season','event','fixture','total_points',
                            'minutes','goals_scored','assists','team_elo','opp_team_elo','position','bonus','bps',
                            'clean_sheets','goals_conceded','was_home','expected_assists','expected_goal_involvements',
                            'expected_goals','expected_goals_conceded','starts',
                            'cbi','defensive_contribution','recoveries','tackles',
                            'saves','team_name','opp_team_name']]

    table_name = "playergw"
    db_creator.create_staging_table_then_insert_data(table_name, data=playergw)
    playergwdf = db_creator.table_to_df(table_name=table_name)
    print("Final playergw dataframe loaded into psql!")
    print(playergwdf.info())

Loading season 20182019...
Loading season 20192020...
Loading season 20202021...
Loading season 20212022...
Loading season 20222023...
Loading season 20232024...
Loading season 20252026...
Loading season 2024-25 (manual merge)...
Loading season 20262027 (API)...
Loaded data for element_id: 1/584
Loaded data for element_id: 2/584
Loaded data for element_id: 3/584
Loaded data for element_id: 4/584
Loaded data for element_id: 5/584
Loaded data for element_id: 6/584
Loaded data for element_id: 7/584
Loaded data for element_id: 8/584
Loaded data for element_id: 9/584
Loaded data for element_id: 10/584
Loaded data for element_id: 11/584
Loaded data for element_id: 12/584
Loaded data for element_id: 13/584
Loaded data for element_id: 14/584
Loaded data for element_id: 15/584
Loaded data for element_id: 16/584
Loaded data for element_id: 17/584
Loaded data for element_id: 18/584
Loaded data for element_id: 19/584
Loaded data for element_id: 20/584
Loaded data for element_id: 21/584
Loaded data

c:\Users\Chris\OneDrive\Documents\Coding\FPL-Points-Predictor\Load_Merge_FPL_Data.py:294: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_merged = pd.concat(list(self.merged_data.values()), axis=0)
c:\Users\Chris\OneDrive\Documents\Coding\FPL-Points-Predictor\Load_Merge_FPL_Data.py:297: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_rest_of_current_season = pd.concat([all_merged, current_year_players_allfixtures], axis=0)



Total records: 229872
Final playergw dataframe loaded into psql!
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 229872 entries, 0 to 229871
Data columns (total 30 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   player_name_id              229872 non-null  object 
 1   element                     229872 non-null  float64
 2   value                       207680 non-null  float64
 3   season                      229872 non-null  float64
 4   event                       229872 non-null  float64
 5   fixture                     229872 non-null  float64
 6   total_points                207680 non-null  float64
 7   minutes                     207680 non-null  float64
 8   goals_scored                207680 non-null  float64
 9   assists                     207680 non-null  float64
 10  team_elo                    229872 non-null  float64
 11  opp_team_elo                229872 non-null  float64
 12  positi

In [8]:
filtered = playergwdf['player_name_id'] == "Mohamed Salah"
playergwdf[filtered].pivot_table(index='season',values='expected_goals',aggfunc='count')

,expected_goals
season,
20182019.0,0
20192020.0,0
20202021.0,0
20212022.0,0
20222023.0,38
20232024.0,38
20242025.0,38
20252026.0,38


In [14]:
filtered = (playergwdf['event']==1) & (playergwdf['player_name_id']=="Jordan Pickford")
playergwdf[filtered].sort_values(by='season')


,player_name_id,element,value,season,event,fixture,total_points,minutes,goals_scored,assists,...,expected_goals,expected_goals_conceded,starts,cbi,defensive_contribution,recoveries,tackles,saves,team_name,opp_team_name
267,Jordan Pickford,154.0,50.0,20182019.0,1.0,10.0,1.0,90.0,0.0,0.0,...,NaN,NaN,NaN,1.0,NaN,9.0,0.0,2.0,Everton,Wolves
22055,Jordan Pickford,148.0,55.0,20192020.0,1.0,4.0,7.0,90.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,Everton,Crystal Palace
44506,Jordan Pickford,157.0,50.0,20202021.0,1.0,4.0,8.0,90.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,Everton,Spurs
69153,Jordan Pickford,170.0,50.0,20212022.0,1.0,4.0,2.0,90.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,Everton,Southampton
94517,Jordan Pickford,182.0,45.0,20222023.0,1.0,3.0,3.0,90.0,0.0,0.0,...,0.0,0.00,0.0,NaN,NaN,NaN,NaN,5.0,Everton,Chelsea
121108,Jordan Pickford,263.0,45.0,20232024.0,1.0,5.0,2.0,90.0,0.0,0.0,...,0.0,1.50,1.0,NaN,NaN,NaN,NaN,1.0,Everton,Fulham
180518,Jordan Pickford,235.0,50.0,20242025.0,1.0,3.0,1.0,90.0,0.0,0.0,...,0.0,1.43,1.0,NaN,NaN,NaN,NaN,2.0,Everton,Brighton
150955,Jordan Pickford,287.0,55.0,20252026.0,1.0,10.0,2.0,90.0,0.0,0.0,...,0.0,2.07,1.0,3.0,0.0,8.0,0.0,2.0,Everton,Leeds
216800,Jordan Pickford,226.0,NaN,20262027.0,1.0,3.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Everton,Crystal Palace


In [10]:
playergwdf.to_csv('playergwdf.csv',index=False)

# Get FPL Player current status

In [11]:
#Create DF from FPL API Request
headers={'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36'}
get=requests.get("https://fantasy.premierleague.com/api/bootstrap-static/", headers=headers, timeout=5)
data=json.loads(get.text)

#player_raw = Summary stats for each player for the current season
cols=list(data['elements'][0].keys())
player_raw = pd.DataFrame(data['elements'], columns = cols)
player = player_raw[['web_name','id','now_cost','status','news','chance_of_playing_next_round','chance_of_playing_this_round']]
player['season']=current_season
player['datetime_now'] = pd.to_datetime('now')

table_name = "player_status"
if __name__ == "__main__":
    db_creator.create_staging_table_then_insert_data(table_name, data=player)
    player_status = db_creator.table_to_df(table_name=table_name)

C:\Users\Chris\AppData\Local\Temp\ipykernel_4920\3829675733.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  player['season']=current_season
C:\Users\Chris\AppData\Local\Temp\ipykernel_4920\3829675733.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  player['datetime_now'] = pd.to_datetime('now')


In [12]:
finish=time.perf_counter()
print(f'Finished in {round(((finish-start)/60),2)} minute(s)')
#Finished in 6.05 minute(s)

Finished in 11.91 minute(s)


In [13]:
# Notification when file has finished running
from plyer import notification

notification.notify(
    title='Script Complete',
    message='Your file has finished running!',
    app_name='Jupyter Notebook',
    timeout=20  # notification stays for 20 seconds
)